# Fluid Mechanics
For experimenting with fluid mechanics and validating that it works

In [ ]:
from dataclasses import dataclass

## Simple

In [ ]:
@dataclass
class BlowdownParameters:
    """Masses are calculated using the parameters"""
    P0: float  # Initial pressure [Pa]
    V0: float  # Initial volume [m3]
    T0: float  # Initial temperature [K]

    rho_fuel: float = 1000.0  # Density of the fuel [kg/m3] (e.g. water)

    # https://www.engineeringtoolbox.com/individual-universal-gas-constant-d_588.html
    R_pres: float = 8.314  # Pressurant gas constant [J/(kg*K)]

@dataclass
class BlowdownState:
    m_fuel: float # [kg]
    m_pres: float # [kg]
    V_fuel: float # [m3]
    V_press: float # [m3]
    P: float  # [Pa]


class BlowdownSystem:
    def __init__(self, params: BlowdownParameters):
        """Blowdown System with initial parameters. Assumes 10% ullage

        Parameters
        ----------
        P0 : float
            Initial pressure (Pa)
        V0 : float
            Initial volume (m3)
        T0 : float
            Initial temperature (K)
        gamma : float, optional
            Pressure specific heat ratio (default of 1.4 like for air)
        """
        self.P0 = params.P0
        self.V0 = params.V0
        self.T0 = params.T0
        self.rho_fuel = params.rho_fuel
        self.R_pres = params.R_pres

        self.m_pres = self.P0 * self.V0 / (self.R_pres * self.T0) # ideal gas law
        self.m_fuel = self.rho_fuel * (self.V0 * 0.9)
    
    def propagate(self, m_dot: float, dt: float):
        """Deplete a certain amount of fuel

        Parameters
        ----------
        m_dot : float
            Mass flow rate of fuel (kg/s)
        dt : float
            Time step (s)
        """
        dm = m_dot * dt
        self.m_fuel -= dm

    @property
    def pressure(self):
        """
        Calculate the current pressure in the system using the ideal gas law
        
        Assumptions:
        - Fuel is incompressible
        - The pressurant gas is ideal
        """

        # Fuel and pressurant volumes
        V_fuel = self.m_fuel / self.rho_fuel 
        V_pres = self.V0 - V_fuel
        
        P_pres = self.m_pres * self.R_pres * self.T0 / V_pres

        return BlowdownState(
            m_fuel=self.m_fuel, # this changes, but is a class variable
            m_pres=self.m_pres, # this is always constant
            V_fuel=V_fuel,  # these are calculated
            V_press=V_pres, # these are calculated
            P=P_pres # this is also calculated
        )
    

# TODO: test this with a simple valve and see how mass flow rate changes with pressure drop